<a href="https://colab.research.google.com/github/Rich-sam/-sentiment-analysis-model/blob/main/project_flood_dashboards.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# auto_flood_forecast.py
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import sqlite3

# ------------------------------
# 1. CONFIGURATION (Yala Basin)
# ------------------------------
THRESHOLDS = {
    "alert": 50,    # m3/s -> Alert Level
    "warning": 100  # m3/s -> Warning Level
}

# Simulated WRF rainfall-runoff input (replace with your actual model output)
def fetch_forecast_from_model():
    """
    In reality, this reads WRF rainfall grids -> runs hydrological model.
    Here we use your provided tables as example.
    """
    data = {
        "date": pd.date_range("2026-04-27", periods=7),
        "rainfall_mm": [20, 32, 20, 40, 15, 10, 5],
        "forecast_flow_m3s": [100, 150, 155, 240, 230, 180, 100]
    }
    df = pd.DataFrame(data)
    df["date"] = pd.to_datetime(df["date"])
    return df

# ------------------------------
# 2. ALERT LOGIC
# ------------------------------
def classify_risk(row):
    flow = row["forecast_flow_m3s"]
    if flow >= THRESHOLDS["warning"]:
        return "HIGH"
    elif flow >= THRESHOLDS["alert"]:
        return "MEDIUM"
    else:
        return "LOW"

def generate_alert(df):
    """
    Returns alert message if any HIGH risk day exists.
    """
    high_risk_days = df[df["risk"] == "HIGH"]
    if not high_risk_days.empty:
        first_high = high_risk_days.iloc[0]["date"].strftime("%d-%b")
        peak_flow = high_risk_days["forecast_flow_m3s"].max()
        msg = (f"FLOOD ALERT: HIGH risk from {first_high}. "
               f"Peak flow ~{peak_flow:.0f} m³/s. "
               f"Monitor Yala & Nyando basins.")
        return msg
    else:
        return "No high flood risk in forecast."

# ------------------------------
# 3. STORE DATA (for dashboard)
# ------------------------------
def save_to_database(df):
    conn = sqlite3.connect("flood_forecast.db")
    df.to_sql("forecast", conn, if_exists="replace", index=False)
    # Also log generation time
    meta = pd.DataFrame({"update_time": [datetime.now()]})
    meta.to_sql("metadata", conn, if_exists="replace", index=False)
    conn.close()

# ------------------------------
# 4. MAIN EXECUTION (automate via cron/scheduler)
# ------------------------------
def run_forecast_pipeline():
    print(f"[{datetime.now()}] Running flood forecast pipeline...")

    # Step 1: Get model forecast
    df = fetch_forecast_from_model()

    # Step 2: Assign risk levels
    df["risk"] = df.apply(classify_risk, axis=1)

    # Step 3: Generate alert message
    alert_msg = generate_alert(df)
    print("ALERT MESSAGE:", alert_msg)

    # Step 4: Save for dashboard
    save_to_database(df)

    # Optional: send email/SMS here (using smtplib or Twilio)

    print("Pipeline complete.")
    return df, alert_msg

if __name__ == "__main__":
    run_forecast_pipeline()

[2026-04-29 08:31:47.804366] Running flood forecast pipeline...
ALERT MESSAGE: FLOOD ALERT: HIGH risk from 27-Apr. Peak flow ~240 m³/s. Monitor Yala & Nyando basins.
Pipeline complete.


In [6]:
import streamlit as st
import pandas as pd
import sqlite3
import plotly.graph_objects as go
from datetime import datetime

st.set_page_config(page_title="Yala & Nyando Flood EWS", layout="wide")

# ------------------------------
# Load latest forecast data
# ------------------------------
@st.cache_data(ttl=3600)  # refresh every hour
def load_forecast():
    conn = sqlite3.connect("flood_forecast.db")
    df = pd.read_sql("SELECT * FROM forecast ORDER BY date", conn)
    # Convert 'date' column to datetime objects
    df['date'] = pd.to_datetime(df['date'])
    meta = pd.read_sql("SELECT update_time FROM metadata", conn)
    conn.close()
    last_update = meta.iloc[0,0] if not meta.empty else "Unknown"
    return df, last_update

df, last_update = load_forecast()

# ------------------------------
# Page Header
# ------------------------------
st.title("🌊 Flood Early Warning System – Yala & Nyando Basins")
st.caption(f"Last forecast update: {last_update} | Valid: {df['date'].min().date()} to {df['date'].max().date()}")

# ------------------------------
# Key Metrics
# ------------------------------
col1, col2, col3, col4 = st.columns(4)
col1.metric("Peak Flow (7d)", f"{df['forecast_flow_m3s'].max():.0f} m³/s")
col2.metric("Alert Threshold", "50 m³/s")
col3.metric("Warning Threshold", "100 m³/s")
highest_risk = df[df["risk"] == "HIGH"].shape[0]
col4.metric("High Risk Days", highest_risk, delta="⚠️" if highest_risk>0 else "✅")

# ------------------------------
# Hydrograph with Thresholds
# ------------------------------
st.subheader("📈 River Flow Forecast vs. Alert/Warning Levels")

fig = go.Figure()
fig.add_trace(go.Scatter(x=df["date"], y=df["forecast_flow_m3s"],
                         mode="lines+markers", name="Forecast Flow"))
fig.add_hline(y=50, line_dash="dash", line_color="orange",
              annotation_text="Alert (50 m³/s)")
fig.add_hline(y=100, line_dash="dash", line_color="red",
              annotation_text="Warning (100 m³/s)")
fig.update_layout(xaxis_title="Date", yaxis_title="Discharge (m³/s)",
                  height=500)
st.plotly_chart(fig, use_container_width=True)

# ------------------------------
# Risk Table & Alert Box
# ------------------------------
colA, colB = st.columns([2,1])
with colA:
    st.subheader("📅 Daily Risk Assessment")
    display_df = df.copy()
    display_df["date"] = display_df["date"].dt.strftime("%d-%b")
    display_df = display_df.rename(columns={
        "date": "Date", "rainfall_mm": "Rain (mm)",
        "forecast_flow_m3s": "Flow (m³/s)", "risk": "Risk"
    })
    # Colour risk cells
    def color_risk(val):
        if val == "HIGH":
            return "background-color: #ffcccc"
        elif val == "MEDIUM":
            return "background-color: #fff3cd"
        return ""
    st.dataframe(display_df.style.applymap(color_risk, subset=["Risk"]), use_container_width=True)

with colB:
    st.subheader("🚨 Active Alerts")
    if highest_risk > 0:
        st.error("⚠️ **HIGH FLOOD RISK** – Take action now. Avoid river crossings.")
    elif (df["risk"] == "MEDIUM").any():
        st.warning("🟡 **MEDIUM RISK** – Stay alert, monitor updates.")
    else:
        st.success("✅ **LOW RISK** – Routine monitoring only.")

# ------------------------------
# Map Placeholder (you can overlay rainfall grid)
# ------------------------------
st.subheader("🗺️ Flood Risk Map – Yala Basin (24h Rainfall Forecast)")
st.image("https://via.placeholder.com/800x400?text=Insert+your+WRF+rainfall+map+here",
         caption="Example: Rainfall intensity over sub-catchments (from WRF model)")
st.info("Integrate actual GeoTIFF or shapefile overlay for production.")

# Footer
st.markdown("---")
st.caption("Data from WRF rainfall-runoff model | Thresholds based on KMD guidelines")

2026-04-29 08:36:15.795 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-29 08:36:15.797 No runtime found, using MemoryCacheStorageManager
2026-04-29 08:36:15.798 No runtime found, using MemoryCacheStorageManager
2026-04-29 08:36:15.799 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-29 08:36:15.809 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-29 08:36:15.810 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-29 08:36:15.811 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-29 08:36:15.816 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-29 08:36:15.817 Thread 'MainThread': missing ScriptRunContext! This warning can be ignor

DeltaGenerator()

In [8]:
# Hardcoded from your second image
df = pd.DataFrame([
    ["2026-04-27", 20, 100],
    ["2026-04-28", 32, 150],
    ["2026-04-29", 20, 155],
    ["2026-04-30", 40, 240],
    ["2026-05-01", 15, 230],
    ["2026-05-02", 10, 180],
    ["2026-05-03", 5, 100]
], columns=["date", "rainfall_mm", "forecast_flow_m3s"])
df["date"] = pd.to_datetime(df["date"], format="%Y-%m-%d")

In [9]:
%%writefile app.py
import streamlit as st
import pandas as pd
import sqlite3
import plotly.graph_objects as go
from datetime import datetime

st.set_page_config(page_title="Yala & Nyando Flood EWS", layout="wide")

# ------------------------------
# Load latest forecast data
# ------------------------------
@st.cache_data(ttl=3600)  # refresh every hour
def load_forecast():
    conn = sqlite3.connect("flood_forecast.db")
    df = pd.read_sql("SELECT * FROM forecast ORDER BY date", conn)
    # Convert 'date' column to datetime objects
    df['date'] = pd.to_datetime(df['date'])
    meta = pd.read_sql("SELECT update_time FROM metadata", conn)
    conn.close()
    last_update = meta.iloc[0,0] if not meta.empty else "Unknown"
    return df, last_update

df, last_update = load_forecast()

# ------------------------------
# Page Header
# ------------------------------
st.title("🌊 Flood Early Warning System – Yala & Nyando Basins")
st.caption(f"Last forecast update: {last_update} | Valid: {df['date'].min().date()} to {df['date'].max().date()}")

# ------------------------------
# Key Metrics
# ------------------------------
col1, col2, col3, col4 = st.columns(4)
col1.metric("Peak Flow (7d)", f"{df['forecast_flow_m3s'].max():.0f} m³/s")
col2.metric("Alert Threshold", "50 m³/s")
col3.metric("Warning Threshold", "100 m³/s")
highest_risk = df[df["risk"] == "HIGH"].shape[0]
col4.metric("High Risk Days", highest_risk, delta="⚠️" if highest_risk>0 else "✅")

# ------------------------------
# Hydrograph with Thresholds
# ------------------------------
st.subheader("📈 River Flow Forecast vs. Alert/Warning Levels")

fig = go.Figure()
fig.add_trace(go.Scatter(x=df["date"], y=df["forecast_flow_m3s"],
                         mode="lines+markers", name="Forecast Flow"))
fig.add_hline(y=50, line_dash="dash", line_color="orange",
              annotation_text="Alert (50 m³/s)")
fig.add_hline(y=100, line_dash="dash", line_color="red",
              annotation_text="Warning (100 m³/s)")
fig.update_layout(xaxis_title="Date", yaxis_title="Discharge (m³/s)",
                  height=500)
st.plotly_chart(fig, use_container_width=True)

# ------------------------------
# Risk Table & Alert Box
# ------------------------------
colA, colB = st.columns([2,1])
with colA:
    st.subheader("📅 Daily Risk Assessment")
    display_df = df.copy()
    display_df["date"] = display_df["date"].dt.strftime("%d-%b")
    display_df = display_df.rename(columns={
        "date": "Date", "rainfall_mm": "Rain (mm)",
        "forecast_flow_m3s": "Flow (m³/s)", "risk": "Risk"
    })
    # Colour risk cells
    def color_risk(val):
        if val == "HIGH":
            return "background-color: #ffcccc"
        elif val == "MEDIUM":
            return "background-color: #fff3cd"
        return ""
    st.dataframe(display_df.style.applymap(color_risk, subset=["Risk"]), use_container_width=True)

with colB:
    st.subheader("🚨 Active Alerts")
    if highest_risk > 0:
        st.error("⚠️ **HIGH FLOOD RISK** – Take action now. Avoid river crossings.")
    elif (df["risk"] == "MEDIUM").any():
        st.warning("🟡 **MEDIUM RISK** – Stay alert, monitor updates.")
    else:
        st.success("✅ **LOW RISK** – Routine monitoring only.")

# ------------------------------
# Map Placeholder (you can overlay rainfall grid)
# ------------------------------
st.subheader("🗺️ Flood Risk Map – Yala Basin (24h Rainfall Forecast)")
st.image("https://via.placeholder.com/800x400?text=Insert+your+WRF+rainfall+map+here",
         caption="Example: Rainfall intensity over sub-catchments (from WRF model)")
st.info("Integrate actual GeoTIFF or shapefile overlay for production.")

# Footer
st.markdown("---")
st.caption("Data from WRF rainfall-runoff model | Thresholds based on KMD guidelines")


Writing app.py


In [ ]:
!streamlit run app.py



2026-04-29 08:39:13.118 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.236.150.183:8501

